# Multi-seed consolidation of the mechanism experiments (seeds 13, 17)

The core realizable gain is already 3-seed. This extends the two **mechanism** results — the
ones that engage the rebutted literature — to seeds 13 & 17 so every load-bearing claim is
multi-seed before writing:

- **Directionality / topology-specificity** (`L_upQrev`, `L_upQrand`) — resolves the unresolved
  seed-11 forward−reversed gap (+0.008, p=0.19).
- **k=2 graph-robustness** (`L_upQ_k2`, `L_upQpred_k2`) — confirms the over-connectivity defense
  isn't a single-seed artifact.

Pre-reg: `preregistration_multiseed_mechanism.md`. All prerequisites (L baselines, per-seed
fullspan evals, k=2 + reversed/random edge sets) are already on Drive/committed.

**Idempotent — Runtime → T4 GPU → Run all.** Skips any run already complete.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config (SEEDS = [13, 17])

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # blank -> auto-detect
SEEDS=[13,17]  # seed 11 already done; these consolidate the mechanism results
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; print('Found',c); break
    else: raise RuntimeError('set DRIVE_CAMELS_PATH')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydrology_runs'; os.makedirs(DRIVE_RUNS,exist_ok=True)
print('seeds', SEEDS)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(os.path.join(REPO_DIR,'runs','topology_ablation','component0'),exist_ok=True)
print('symlinks ready')

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Build all features (seed-independent + per-seed)

Reversed/random/k2-oracle are observed-Q (seed-independent). k2-predicted needs each seed's
fullspan predictions. Idempotent: skips builds already present with a 'date'-named index.

In [ ]:
%cd {REPO_DIR}
import pickle
FEAT='experiments/topology_ablation/features'
def named_ok(p):
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb')); return d[next(iter(d))].index.name=='date'
!python experiments/topology_ablation/generate_topology_attributes.py 2>&1 | tail -1
# forward observed (Δ ref feature) + reversed + random (all seed-independent observed-Q)
if not named_ok(f'{FEAT}/upstream_q_component0_lag1.p'):
    !python experiments/topology_ablation/build_upstream_discharge_feature.py --network component0 --lag-days 1 2>&1 | tail -1
if not (named_ok(f'{FEAT}/upstream_q_reversed_component0_lag1.p') and named_ok(f'{FEAT}/upstream_q_random_component0_lag1.p')):
    !python experiments/topology_ablation/build_directionality_variants.py --network component0 --lag-days 1 2>&1 | tail -3
# k2 OBSERVED oracle feature (seed-independent) — rebuild from committed k2 edge set
import numpy as np, pandas as pd, networkx as nx
from pathlib import Path
def build_k2_obs():
    P1=Path('topology_analysis/phase1_network_discovery/outputs')
    BASE=Path('runs/topology_ablation/component0'); TOPO=Path('datasets/camels_us/camels_attributes_v2.0/camels_topo.txt')
    pr=pd.read_csv(P1/'component0_edges_k2.csv',dtype={'parent_id':str,'child_id':str})
    basins=[l.strip() for l in open(P1/'component0_basins.txt') if l.strip()]
    area=pd.read_csv(TOPO,sep=';',dtype={'gauge_id':str}).set_index('gauge_id')['area_gages2'].to_dict()
    fs=pickle.load(open(BASE/'_Lfullspan_eval_seed11/test/model_epoch030/test_results.p','rb'))
    obs={b:pd.Series(d['1D']['xr']['QObs(mm/d)_obs'].values.squeeze(),index=pd.DatetimeIndex(pd.to_datetime(d['1D']['xr']['date'].values),name='date')) for b,d in fs.items()}
    G=nx.DiGraph(); G.add_nodes_from(basins)
    for _,r in pr.iterrows():
        if r['parent_id'] in basins and r['child_id'] in basins: G.add_edge(r['parent_id'],r['child_id'])
    feats={}
    for b in basins:
        if b not in obs: continue
        idx=obs[b].index; par=list(G.predecessors(b))
        if not par: feats[b]=pd.DataFrame({'upstream_q':np.zeros(len(idx))},index=idx); continue
        agg=pd.Series(0.0,index=idx); w=0.0
        for p in par:
            if p not in obs: continue
            pa=float(area.get(p,0.0)); agg=agg.add((obs[p].reindex(idx)*pa).fillna(0.0),fill_value=0.0); w+=pa
        if w>0: agg=agg/w
        feats[b]=pd.DataFrame({'upstream_q':agg.shift(1).fillna(0.0).values},index=idx)
    pickle.dump(feats,open(f'{FEAT}/upstream_q_obs_component0_k2_lag1.p','wb'))
    print('k2 obs feature rebuilt')
if not named_ok(f'{FEAT}/upstream_q_obs_component0_k2_lag1.p'): build_k2_obs()
print('seed-independent features ready')

## Cell 8 — Per-seed sweep (ensure L; build k2-predicted; train 4 conditions)

For each seed: ensure L baseline exists (Δ ref), build that seed's k2-predicted feature from its
fullspan eval, then train reversed / random / k2-oracle / k2-realizable. All idempotent.

In [ ]:
%cd {REPO_DIR}
import glob, pickle, numpy as np, pandas as pd, networkx as nx
from pathlib import Path
B=f'{REPO_DIR}/runs/topology_ablation/component0'
BASIN='topology_analysis/phase1_network_discovery/outputs/component0_basins.txt'
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
def build_k2_pred(seed):
    P1=Path('topology_analysis/phase1_network_discovery/outputs'); TOPO=Path('datasets/camels_us/camels_attributes_v2.0/camels_topo.txt')
    pr=pd.read_csv(P1/'component0_edges_k2.csv',dtype={'parent_id':str,'child_id':str})
    basins=[l.strip() for l in open(P1/'component0_basins.txt') if l.strip()]
    area=pd.read_csv(TOPO,sep=';',dtype={'gauge_id':str}).set_index('gauge_id')['area_gages2'].to_dict()
    fs=pickle.load(open(f'{B}/_Lfullspan_eval_seed{seed}/test/model_epoch030/test_results.p','rb'))
    pred={b:pd.Series(d['1D']['xr']['QObs(mm/d)_sim'].values.squeeze(),index=pd.DatetimeIndex(pd.to_datetime(d['1D']['xr']['date'].values),name='date')) for b,d in fs.items()}
    G=nx.DiGraph(); G.add_nodes_from(basins)
    for _,r in pr.iterrows():
        if r['parent_id'] in basins and r['child_id'] in basins: G.add_edge(r['parent_id'],r['child_id'])
    feats={}
    for b in basins:
        if b not in pred: continue
        idx=pred[b].index; par=list(G.predecessors(b))
        if not par: feats[b]=pd.DataFrame({'upstream_q':np.zeros(len(idx))},index=idx); continue
        agg=pd.Series(0.0,index=idx); w=0.0
        for p in par:
            if p not in pred: continue
            pa=float(area.get(p,0.0)); agg=agg.add((pred[p].reindex(idx)*pa).fillna(0.0),fill_value=0.0); w+=pa
        if w>0: agg=agg/w
        feats[b]=pd.DataFrame({'upstream_q':agg.shift(1).fillna(0.0).values},index=idx)
    out=f'{FEAT}/upstream_q_pred_component0_k2_seed{seed}_lag1.p'; pickle.dump(feats,open(out,'wb')); return out

def rn(cond, feat, s):
    if done(cond,s): print(f'[skip] {cond} seed{s}'); return
    !python experiments/topology_ablation/run_upstream_feature.py --network component0 --seed {s} --device cuda:0 --feature-file {feat} --cond-name {cond} 2>&1 | tail -2

for s in SEEDS:
    print(f'\n########## SEED {s} ##########')
    # ensure L baseline
    if not done('L',s):
        Ld=f'{B}/L_component0_seed{s}'
        !python experiments/topology_ablation/make_configs.py --network component0 --basin-file {BASIN} --seed {s} --device cuda:0 --epochs 30
        !python neuralhydrology/nh_run.py train --config-file experiments/topology_ablation/configs/L_component0_seed{s}.yaml 2>&1 | tail -2
        ts=sorted(glob.glob(f'{Ld}_*'));
        if ts: os.rename(ts[-1],Ld)
        !python neuralhydrology/nh_run.py evaluate --run-dir {Ld} --epoch 30 2>&1 | tail -1
    else: print(f'[skip] L seed{s}')
    # directionality (seed-independent features)
    rn('L_upQrev', f'{FEAT}/upstream_q_reversed_component0_lag1.p', s)
    rn('L_upQrand', f'{FEAT}/upstream_q_random_component0_lag1.p', s)
    # k2 oracle (seed-independent obs feature)
    rn('L_upQ_k2', f'{FEAT}/upstream_q_obs_component0_k2_lag1.p', s)
    # k2 realizable (per-seed predicted feature)
    if not done('L_upQpred_k2',s):
        pf=build_k2_pred(s); rn('L_upQpred_k2', pf, s)
    else: print(f'[skip] L_upQpred_k2 seed{s}')
    print(f'seed {s} done')

## Cell 9 — Verdict: 3-seed directionality + 3-seed k2 (pooled)

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np, pickle
from scipy.stats import wilcoxon
B=f'{REPO_DIR}/runs/topology_ablation/component0'; ALL=[11]+SEEDS
def nse(cond,s):
    p=f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
feat=pickle.load(open(f'{FEAT}/upstream_q_component0_lag1.p','rb'))
conn=[b for b in feat if np.abs(feat[b]['upstream_q'].values).mean()>0]
featk=pickle.load(open(f'{FEAT}/upstream_q_obs_component0_k2_lag1.p','rb'))
connk=[b for b in featk if np.abs(featk[b]['upstream_q'].values).mean()>0]
def pooled_delta(cond, cset):
    d=[]
    for s in ALL:
        x=nse(cond,s); L=nse('L',s)
        if x is None or L is None: continue
        c=[b for b in cset if b in x.index and b in L.index]; d+= list((x.loc[c]-L.loc[c]).values)
    d=np.array(d); p=wilcoxon(d,alternative='greater').pvalue if len(d)>=6 and np.any(d!=0) else float('nan')
    return np.median(d), p, len(d)
def pooled_pair(a,b,cset):
    d=[]
    for s in ALL:
        xa,xb=nse(a,s),nse(b,s)
        if xa is None or xb is None: continue
        c=[k for k in cset if k in xa.index and k in xb.index]; d+= list((xa.loc[c]-xb.loc[c]).values)
    d=np.array(d); return (np.median(d), wilcoxon(d,alternative='greater').pvalue, len(d)) if len(d) else (float('nan'),float('nan'),0)
print('=== DIRECTIONALITY (pooled over available seeds, forward-connected) ===')
for cond,lbl in [('L_upQ','forward'),('L_upQrev','reversed'),('L_upQrand','random')]:
    m,p,n=pooled_delta(cond,conn); print(f'  {lbl:<10} median Δ={m:+.4f}  p={p:.1e}  (n={n})')
fr=pooled_pair('L_upQ','L_upQrev',conn); frn=pooled_pair('L_upQ','L_upQrand',conn)
print(f'  forward-reversed (paired) Δ={fr[0]:+.4f} p={fr[1]:.1e}  <- the resolution')
print(f'  forward-random   (paired) Δ={frn[0]:+.4f} p={frn[1]:.1e}')
print('\n=== k=2 GRAPH-ROBUSTNESS (pooled, k2-connected) ===')
for cond,lbl in [('L_upQpred','full realizable'),('L_upQpred_k2','k2 realizable'),('L_upQ_k2','k2 oracle')]:
    m,p,n=pooled_delta(cond,connk); print(f'  {lbl:<16} median Δ={m:+.4f}  p={p:.1e}  (n={n})')
print('\nReport this block back for filing.')

## Cell 10 — Persistence check

In [ ]:
%cd {REPO_DIR}
print('=== persistence (in Drive?) ===')
for s in SEEDS:
    for cond in ['L_upQrev','L_upQrand','L_upQ_k2','L_upQpred_k2']:
        dp=f'{DRIVE_RUNS}/topology_ablation/component0/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
        print(f'  {cond}_seed{s}: {os.path.isfile(dp)}')

## Done

8 new runs persist to Drive (4 conditions × 2 seeds). Report the Cell 9 verdict + Cell 10
persistence back; then the mechanism results are fully 3-seed and writing is unblocked.